# RecetIA v2 — Asistente de recetas con RAG sobre todo el dataset (~20k)

En este notebook montamos el asistente completo usando **todas las recetas válidas** de `somosnlp/RecetasDeLaAbuela`, búsqueda semántica con ChromaDB, filtros automáticos de alérgenos/dieta, generación con verificación inteligente en `qwen3:1.7b` (Ollama) y una interfaz web en Flask.

### Requisitos
- Librerías: `pip install datasets pandas sentence-transformers chromadb ollama flask`
- Ollama en local con el modelo descargado: `ollama pull qwen3:1.7b`

## 1. Carga, limpieza completa y etiquetado

Cargamos las ~20.000 recetas completas y nos aseguramos de que **todas** tengan los campos esenciales (`Nombre`, `Ingredientes` y `Pasos`) rellenos y sin duplicados. También limpiamos el formato de listas y rellenamos `Categoria` y `Dificultad` cuando vengan vacías para no tener valores nulos en ningún campo.

In [1]:
import ast
from datasets import load_dataset
import pandas as pd

# Cargamos las ~20.200 recetas desde Hugging Face
ds = load_dataset("somosnlp/RecetasDeLaAbuela", "version_1", split="train")
df = ds.to_pandas()

# Nos quedamos solo con las 6 columnas útiles
columnas = ["Nombre", "Ingredientes", "Pasos", "Categoria", "Dificultad", "Valor nutricional"]
df = df[columnas].copy()

# Aseguramos que todas las recetas tengan Nombre, Ingredientes y Pasos reales
# (algunas filas traen texto "[]" o "None" en vez de nulo real)
for col in ["Nombre", "Ingredientes", "Pasos"]:
    df = df[df[col].notna()]
    df = df[~df[col].astype(str).str.strip().isin(["", "[]", "None", "nan"])]

# Descartamos recetas incompletas con menos de 10 caracteres y quitamos nombres repetidos
df = df[df["Ingredientes"].astype(str).str.len() >= 10]
df = df[df["Pasos"].astype(str).str.len() >= 10]
df = df.drop_duplicates(subset=["Nombre"]).reset_index(drop=True)

# El dataset guarda ingredientes, pasos y categorías como listas en texto ("['1 cebolla', '2 ajos']").
# ast.literal_eval las convierte en una lista normal de Python para poder unirlas de forma legible.
def formatear_campo(valor, sep=", ", defecto=""):
    if valor is None or pd.isna(valor):
        return defecto
    texto = str(valor).strip()
    if texto in ("", "[]", "None", "nan"):
        return defecto
    if texto.startswith("[") and texto.endswith("]"):
        try:
            elementos = [str(x).strip() for x in ast.literal_eval(texto) if str(x).strip()]
            return sep.join(elementos) if elementos else defecto
        except Exception:
            pass
    return texto

df["Nombre"] = df["Nombre"].astype(str).str.strip()
df["Ingredientes"] = df["Ingredientes"].apply(lambda v: formatear_campo(v, sep=", "))
df["Pasos"] = df["Pasos"].apply(lambda v: formatear_campo(v, sep="\n"))
# Si falta categoría, dificultad o nutrición, ponemos un texto por defecto para tener 0 nulos
df["Categoria"] = df["Categoria"].apply(lambda v: formatear_campo(v, sep=", ", defecto="General"))
df["Dificultad"] = df["Dificultad"].apply(lambda v: formatear_campo(v, defecto="No especificada"))
df["Valor nutricional"] = df["Valor nutricional"].apply(lambda v: formatear_campo(v, defecto="No especificado"))

print("Recetas limpias totales:", df.shape)
print("Nulos por columna:\n", df.isna().sum())
df.head(3)

Recetas limpias totales: (20012, 6)
Nulos por columna:
 Nombre               0
Ingredientes         0
Pasos                0
Categoria            0
Dificultad           0
Valor nutricional    0
dtype: int64


,Nombre,Ingredientes,Pasos,Categoria,Dificultad,Valor nutricional
0,Tacos Dorados de Papa,"½ taza de cilantro finamente picado, 1 taza de...",Pon las papas enteras en una olla mediana y cú...,vegetarianos,No especificada,"Alto en calorías, Alto en grasas, Alto en sodio"
1,Ensalada de Chayotes,"½ cucharadita sal, ¼ cucharadita de pimienta r...",Coloca los chayotes en una cacerola y cúbralos...,vegetarianos,No especificada,"Bajo en calorías, Sin grasa, Alto en fibra"
2,Huevos a la Mexicana,"½ Chile Serrano cortado en pequeños trocitos*,...",Calienta el aceite en una sartén mediana a fue...,vegetarianos,No especificada,"Alto en calorías, Alto en grasas, Alto en sodio"


In [2]:
import re

# Pasamos el texto a minúsculas y quitamos tildes para que "atún" y "atun" coincidan igual
def limpiar_texto(texto, quitar=()):
    texto = str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu"))
    for frase in quitar:
        texto = texto.replace(frase, " ")
    return texto

# Antes de buscar alérgenos, borramos estas frases para evitar falsos positivos
# (por ejemplo, "leche de coco" no lleva lactosa y "harina de maíz" no lleva gluten)
EXCLUSIONES = [
    "sin gluten", "sin lactosa", "sin huevo", "sin harina", "sin carne",
    "miga de pan sin gluten", "pan sin gluten",
    "harina de maiz", "harina de arroz", "harina de garbanzo", "harina de coco", "harina de almendra",
    "leche de soja", "leche de soya", "leche de coco", "leche de almendra", "leche de avena", "leche vegetal", "leche de tigre",
    "manteca de cacao", "manteca vegetal", "tofu de garbanzo", "cortar la leche",
    "albondigas de seitan", "albondigas veganas", "helado vegano", "queso vegano", "carne vegetal",
]

ALERGENOS = {
    "gluten": [
        "harina", "pan", "pan rallado", "picatostes", "baguette", "bollo", "hogaza", "sandwich",
        "trigo", "cebada", "centeno", "avena", "cuscus", "couscous", "seitan", "cerveza",
        "pasta", "macarron", "macarrones", "macerrones", "espagueti", "spaghetti", "fideo", "tallarin", "tallarines",
        "fusilli", "tortellini", "lasana", "canelon", "canelones", "pasticho",
        "hojaldre", "masa hojaldrada", "masa brisa", "masa para pizza", "empanada", "empanadilla",
        "galleta", "oreo", "bizcocho", "croissant", "medialuna", "crepe", "pizza", "oblea",
        "salsa de soja", "salsa de soya", "sillao",
    ],
    "lactosa": [
        "leche", "queso", "nata", "mantequilla", "manteqeilla", "manteca", "yogur", "yogures", "yogurt", "yoghurt",
        "kefir", "chantilly", "chantilli", "bechamel", "requeson", "ricota", "ricotta", "mascarpone",
        "mozzarella", "mozarella", "parmesano", "cheddar", "manchego", "cabrales", "roquefort",
        "gouda", "emmental", "emental", "edam", "gruyere", "feta", "pecorino", "cotija", "cottage",
        "philadelphia", "fhiladelfia", "fhiladelphia", "nutella", "toblerone", "helado", "pesto",
        "crema de leche", "crema agria", "crema batida", "crema pastelera", "queso crema", "sopa-crema",
    ],
    "huevo": ["huevo", "clara", "yema", "mayonesa", "alioli", "galletas de champana"],
    "frutos_secos": [
        "frutos secos", "nuez", "nueces", "almendra", "avellana", "cacahuete", "cacahuate",
        "mani", "pistacho", "pinon", "pinones", "anacardo", "pecana", "castana", "macadamia",
        "mazapan", "nutella", "pesto",
    ],
    "pescado": [
        "pescado", "atun", "salmon", "bonito", "bacalao", "merluza", "anchoa", "boqueron", "boquerones",
        "caballa", "sardina", "trucha", "lubina", "dorada", "corvina", "lenguado", "besugo",
        "pargo", "cazon", "tilapia", "mojarra", "pejerrey", "chicharro", "cococha", "kokotxa",
        "surimi", "fume", "fumet", "caldo de pescado", "worcestershire", "salsa inglesa",
    ],
    "marisco": [
        "marisco", "gamba", "gambita", "gambon", "gambones", "camaron", "camarones", "camorones",
        "langostino", "cigala", "quisquilla", "cangrejo", "jaiba", "mejillon", "mejillones",
        "almeja", "chirla", "navaja", "ostra", "vieira", "pulpo", "calamar", "calamares",
        "sepia", "sepieta", "chipiron", "chipirones", "caracol", "caracoles",
    ],
    "soja": ["soja", "soya", "sillao", "tofu", "tempeh", "edamame", "miso"],
    "sesamo": ["sesamo", "ajonjoli", "tahini"],
    "mostaza": ["mostaza", "mustard", "dijon"],
    "apio": ["apio"],
}

ANIMAL = (
    ALERGENOS["lactosa"] + ALERGENOS["huevo"] + ALERGENOS["pescado"] + ALERGENOS["marisco"] + [
        "carne", "carnitas", "pollo", "pollito", "cerdo", "cerde", "puerco", "cochino",
        "lechon", "lechona", "res", "ternera", "vaca", "vacuno", "buey", "cordero", "pato",
        "asado de tira", "cuadril", "jamon", "lacon", "serrano", "iberico", "york",
        "tocino", "beicon", "bacon", "panceta", "chorizo", "choricito", "salchicha", "perritos",
        "longaniza", "morcilla", "sobrasada", "mortadela", "peperoni", "butifarron", "butifarrones",
        "lomo", "lomitos", "solomillo", "solomilloo", "entrecot", "chuleta", "costilla", "ribs", "bistec",
        "pechuga", "muslo", "contramuslo", "albondigas", "higado", "mollejas", "oreja", "pate",
        "codillo", "secreto", "sebo", "manteca de cerdo", "avecrem", "caldo de pollo", "caldo de carne",
        "conejo", "liebre", "pavo", "gallina", "perdiz", "codorniz", "sesos", "callos", "mondongo", "tripitas", "tripa",
        "miel", "gelatina", "colapez", "rana",
    ]
)

# Unimos cada lista en un patrón regex (ej: \b(leche|queso|nata)s?\b):
# - \b exige palabra completa (así "pan" no coincide con "panceta" ni "res" con "fresas")
# - s? permite detectar también el plural
# Compilarlo una vez hace que revisar 20.000 recetas tarde menos de 2 segundos
def compilar(palabras):
    grupo = "|".join(palabras)
    return re.compile(rf"\b({grupo})s?\b")

PATRONES_ALERG = {k: compilar(v) for k, v in ALERGENOS.items()}
PATRON_ANIMAL = compilar(ANIMAL)

def detectar(texto, patron, extra_excluir=()):
    texto = limpiar_texto(texto, EXCLUSIONES + list(extra_excluir))
    return bool(patron.search(texto))

COLS_ALERGENOS = list(ALERGENOS.keys())
texto_completo = df["Nombre"] + " " + df["Ingredientes"]

for alergeno, patron in PATRONES_ALERG.items():
    # La manteca de cerdo es grasa animal, no láctea, así que la ignoramos al buscar lactosa
    extra = ("manteca de cerdo",) if alergeno == "lactosa" else ()
    df[alergeno] = texto_completo.apply(lambda t: detectar(t, patron, extra))

# Creamos dos columnas resumen: un booleano rápido y la lista legible de alérgenos
df["tiene_alergenos"] = df[COLS_ALERGENOS].any(axis=1)
df["alergenos"] = df[COLS_ALERGENOS].apply(
    lambda r: ", ".join(c for c in COLS_ALERGENOS if r[c]) or "ninguno",
    axis=1,
)
# El símbolo ~ invierte True/False: es vegana cuando NO detecta ningún ingrediente animal
df["vegano"] = ~texto_completo.apply(lambda t: detectar(t, PATRON_ANIMAL))

print(df[["vegano", "tiene_alergenos"] + COLS_ALERGENOS].mean().round(4))

vegano             0.1637
tiene_alergenos    0.8130
gluten             0.4134
lactosa            0.4830
huevo              0.3094
frutos_secos       0.1074
pescado            0.1313
marisco            0.0820
soja               0.0405
sesamo             0.0158
mostaza            0.0319
apio               0.0271
dtype: float64


## 2. Embeddings y base vectorial con las ~20k recetas

Calculamos los embeddings con `paraphrase-multilingual-MiniLM-L12-v2` (aprovechando GPU si está disponible) y guardamos todo en `./chroma_recetas_20k` por lotes de 1.000 recetas. Si la colección ya tiene todas las recetas cargadas, nos saltamos el indexado para no repetir trabajo.

In [3]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb

# Usamos tarjeta gráfica (CUDA) si hay disponible para que codificar 20k recetas tarde segundos
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dispositivo)
print("Embeddings ejecutándose en:", dispositivo)

# Juntamos título, categoría, dificultad e ingredientes: es lo que el buscador va a "entender"
df["texto"] = (
    "Receta: " + df["Nombre"]
    + ". Categoría: " + df["Categoria"]
    + ". Dificultad: " + df["Dificultad"]
    + ". Ingredientes: " + df["Ingredientes"]
)

# Creamos la base vectorial en disco usando distancia coseno (0 = idéntico, 2 = opuesto)
cliente = chromadb.PersistentClient(path="./chroma_recetas_20k")
col = cliente.get_or_create_collection("recetas_20k", metadata={"hnsw:space": "cosine"})

# Solo calculamos y guardamos los vectores si la base aún no tiene todas las recetas
if col.count() != len(df):
    embeddings = modelo_emb.encode(df["texto"].tolist(), batch_size=128, show_progress_bar=True)
    cols_meta = ["Nombre", "vegano", "tiene_alergenos", "alergenos"] + COLS_ALERGENOS
    documentos = (
        df["texto"] + "\nAlérgenos detectados: " + df["alergenos"] + "\nPasos:\n" + df["Pasos"]
    ).tolist()

    # Subimos en bloques de 1.000 porque ChromaDB tiene un límite máximo por llamada
    for i in range(0, len(df), 1000):
        j = i + 1000
        col.upsert(
            ids=[str(idx) for idx in df.index[i:j]],
            embeddings=embeddings[i:j].tolist(),
            documents=documentos[i:j],
            metadatas=df[cols_meta].iloc[i:j].to_dict(orient="records"),
        )

print("Recetas indexadas en ChromaDB:", col.count())

Embeddings ejecutándose en: cuda
Recetas indexadas en ChromaDB: 20012


## 3. Búsqueda semántica, filtros automáticos y evaluación (`Hit@3`)

Definimos `filtros_de()` para traducir restricciones (*"vegano"*, *"sin gluten"*, *"sin alérgenos"*, etc.) a filtros de ChromaDB, y `buscar()` con búsqueda híbrida sencilla: primero filtra en ChromaDB sobre las 20k recetas limpiando la consulta y luego prioriza entre las candidatas las que contienen las palabras clave pedidas.

In [4]:
# Diccionario que traduce frases del usuario al nombre exacto de la columna en ChromaDB
PATRONES_FILTRO = {
    "sin gluten": "gluten",
    "sin lactosa": "lactosa",
    "sin huevo": "huevo",
    "sin frutos secos": "frutos_secos",
    "sin pescado": "pescado",
    "sin marisco": "marisco",
    "sin soja": "soja",
    "sin sesamo": "sesamo",
    "sin mostaza": "mostaza",
    "sin apio": "apio",
}

# Palabras genéricas que no aportan ingredientes a la hora de reordenar resultados
STOPWORDS = {
    "de", "del", "la", "el", "los", "las", "un", "una", "unos", "unas", "con", "sin", "para",
    "por", "en", "y", "o", "a", "al", "algo", "plato", "receta", "comida", "cena", "almuerzo",
    "desayuno", "facil", "rapida", "rapido", "sencilla", "sencillo", "casera", "casero", "quiero",
}

def filtros_de(pregunta):
    p = limpiar_texto(pregunta)
    condiciones = []
    if "vegan" in p:
        condiciones.append({"vegano": True})
    if "sin alergenos" in p:
        condiciones.append({"tiene_alergenos": False})
    for frase, col_alergeno in PATRONES_FILTRO.items():
        if frase in p:
            condiciones.append({col_alergeno: False})
    if not condiciones:
        return None
    # Si hay más de un filtro a la vez (ej: vegano Y sin gluten), ChromaDB pide combinarlos con "$and"
    return condiciones[0] if len(condiciones) == 1 else {"$and": condiciones}

def limpiar_consulta(pregunta):
    # Quitamos palabras como "sin frutos secos" o "vegano" de la búsqueda por significado,
    # porque de lo contrario el modelo se confunde buscando "frutas" en lugar de "galletas"
    quitar = list(PATRONES_FILTRO.keys()) + ["sin alergenos", "vegano", "vegana", "veganos", "veganas"]
    return " ".join(limpiar_texto(pregunta, quitar).split()) or pregunta

def buscar(pregunta, k=3, filtros=None, mejorar=True):
    # Modo básico (antes): solo busca por similitud de significado
    if not mejorar:
        q = modelo_emb.encode([pregunta]).tolist()
        return col.query(query_embeddings=q, n_results=k, where=filtros)

    # Modo mejorado (después):
    # 1) Limpiamos las palabras del filtro para centrar el vector en el plato/ingrediente
    q_limpia = limpiar_consulta(pregunta)
    q = modelo_emb.encode([q_limpia]).tolist()

    # 2) Pedimos 100 recetas candidatas ya filtradas por alérgenos/vegano
    res = col.query(query_embeddings=q, n_results=100, where=filtros)
    docs, metas, dists = res["documents"][0], res["metadatas"][0], res["distances"][0]

    # 3) Sacamos las palabras clave de la pregunta (quitando conectores como "de", "con", "plato"...)
    palabras = [w for w in q_limpia.split() if w not in STOPWORDS and len(w) > 2]
    if not palabras or not docs:
        return {"documents": [docs[:k]], "metadatas": [metas[:k]], "distances": [dists[:k]]}

    # 4) Reordenamos las 100 candidatas: primero las que tienen la palabra en el título,
    # luego en los ingredientes y, en caso de empate, por menor distancia semántica (-dist)
    def puntuacion(item):
        doc, meta, dist = item
        nombre = limpiar_texto(meta["Nombre"])
        cabecera = limpiar_texto(doc.split("\nPasos:")[0])
        en_nombre = sum(w in nombre for w in palabras)
        en_cabecera = sum(w in cabecera for w in palabras)
        return (en_nombre, en_cabecera, -dist)

    ordenadas = sorted(zip(docs, metas, dists), key=puntuacion, reverse=True)[:k]
    return {
        "documents": [[x[0] for x in ordenadas]],
        "metadatas": [[x[1] for x in ordenadas]],
        "distances": [[x[2] for x in ordenadas]],
    }

# Prueba rápida
consulta_test = "algo vegano con garbanzos"
r = buscar(consulta_test, filtros=filtros_de(consulta_test))
for m, d in zip(r["metadatas"][0], r["distances"][0]):
    print(round(d, 3), m["Nombre"], "| alérgenos:", m["alergenos"])

0.639 garbanzos con pinones | alérgenos: frutos_secos
0.641 quiche lorraine vegana | alérgenos: gluten, frutos_secos, soja
0.536 batido de lentejas | alérgenos: ninguno


### Batería de pruebas mejorada (12 preguntas)

Probamos **12 consultas variadas** comparando **antes** (solo similitud semántica sin filtros ni reordenado) frente a **después** (filtros automáticos + consulta limpia + reordenado por palabras clave).

In [5]:
bateria_pruebas = [
    "Postre sin gluten",
    "Cena rápida con pollo",
    "Receta vegana con lentejas",
    "Arroz con verduras",
    "Sopa caliente para el invierno",
    "Pescado al horno sin lactosa",
    "Bizcocho o tarta sin huevo",
    "Pasta vegana",
    "Ensalada con aguacate y tomate",
    "Galletas sin frutos secos",
    "Comida sencilla sin alergenos",
    "Plato con garbanzos sin marisco",
]

for i, pregunta in enumerate(bateria_pruebas, 1):
    filtro = filtros_de(pregunta)
    res_antes = buscar(pregunta, k=3, filtros=None, mejorar=False)
    res_despues = buscar(pregunta, k=3, filtros=filtro, mejorar=True)

    top_antes = [f"{m['Nombre']} ({m['alergenos']})" for m in res_antes["metadatas"][0]]
    top_despues = [f"{m['Nombre']} ({m['alergenos']})" for m in res_despues["metadatas"][0]]

    print(f"{i:2d}. {pregunta} | filtro: {filtro}")
    print("    Antes:  ", " | ".join(top_antes))
    print("    Después:", " | ".join(top_despues))

 1. Postre sin gluten | filtro: {'gluten': False}
    Antes:   berenjenas rebozadas sin gluten (huevo) | profiteroles sin gluten (lactosa, huevo) | atolillo (lactosa)
    Después: postre borracho (lactosa) | postre economico (lactosa, huevo) | postre con gelatina de pina (lactosa)
 2. Cena rápida con pollo | filtro: None
    Antes:   guiso de gallina (lactosa) | caldo de pollo casero (ninguno) | pollo con durazno chifa (gluten)
    Después: caldo de pollo de la abuela (gluten, apio) | pollo con garbanzos super facil (ninguno) | caldo de pollo sin verduras (ninguno)
 3. Receta vegana con lentejas | filtro: {'vegano': True}
    Antes:   boniato al horno con azucar y canela (ninguno) | revuelto vegano de ajetes (soja) | matambre vegano (gluten)
    Después: batido de lentejas (ninguno) | tacu tacu de lentejas (gluten) | lentejas guisadas (gluten)
 4. Arroz con verduras | filtro: None
    Antes:   verduras esparragadas (gluten) | verduras asadas al horno con patatas (ninguno) | empanada de

### Resumen de resultados de la evaluación

Al pasar de 400 recetas a **20.012 recetas limpias** y combinar los filtros automáticos con el reordenado por palabras clave, la mejora en `Hit@3` es muy clara:

| # | Pregunta | Antes (solo vector) | Después (filtros + reordenado) | Qué pasó |
|---|---|---|---|---|
| 1 | Postre sin gluten | **Sí** | **Sí** | Antes coló *berenjenas* entre los resultados; después los 3 son postres sin gluten. |
| 2 | Cena rápida con pollo | **Sí** | **Sí** | Ambas versiones recuperan recetas de pollo. |
| 3 | Receta vegana con lentejas | **No** | **Sí** | Antes devolvía recetas veganas sin lentejas; después saca *batido de lentejas*, *tacu tacu de lentejas* y *lentejas guisadas*. |
| 4 | Arroz con verduras | **No** | **Sí** | Antes traía platos solo de verduras; después saca *arroz con verduras* directo. |
| 5 | Sopa caliente para el invierno | **Sí** | **Sí** | En ambos casos recupera sopas calientes válidas. |
| 6 | Pescado al horno sin lactosa | **No** | **Sí** | Antes fallaba con *arrollado de pescado* (con lactosa) o frituras; después los 3 son pescado al horno sin lactosa. |
| 7 | Bizcocho o tarta sin huevo | **No** | **Sí** | Antes todas llevaban huevo; después las 3 son tartas sin huevo. |
| 8 | Pasta vegana | **No** | **Sí** | Antes solo devolvía salsas para pasta; después saca 3 platos de pasta vegana. |
| 9 | Ensalada con aguacate y tomate | **No** | **Sí** | Antes faltaba el aguacate; después aparece *ensalada de aguacate y tomate* en primer lugar. |
| 10 | Galletas sin frutos secos | **No** | **No** | Mejora algo (*batido de platano y galletas*), pero sigue sin devolver galletas horneadas como plato principal. |
| 11 | Comida sencilla sin alergenos | **No** | **Sí** | Antes todas tenían alérgenos; después las 3 salen con `ninguno`. |
| 12 | Plato con garbanzos sin marisco | **No** | **Sí** | Antes traía un plato con marisco y sin garbanzos; después aparece *los tradicionales garbanzos guisados*. |

**Conclusión en pocas palabras:**
- **Hit@3 antes:** `3 / 12` (**25,0 %**).
- **Hit@3 después:** `11 / 12` (**91,7 %**).
- **Por qué funciona mejor:** quitar de la consulta palabras como *"sin frutos secos"* o *"sin marisco"* evita que el buscador semántico busque por error recetas de *frutas* o *marisco*, los filtros garantizan las restricciones de alérgenos/vegano, y el reordenado asegura que aparezcan primero las recetas que contienen el ingrediente pedido.
- **Limitación conocida:** el etiquetado por palabras clave sigue siendo aproximado para términos poco comunes o regionales (por ejemplo, palabras como *ponqué* pueden no detectarse como gluten/huevo), aunque sirve bien como filtro base.

## 4. Generación con Qwen3 y verificación inteligente

El asistente responde usando solo las 3 recetas recuperadas. Después aplicamos una **comprobación ligera**: si la respuesta cumple el formato, no menciona ingredientes prohibidos y avisa correctamente de los alérgenos, la devolvemos al momento; **solo si detectamos algún fallo** (respuesta vacía, etiqueta `<think>` residual o falta de aviso de alérgenos), lanzamos una segunda pasada rápida de verificación.

In [6]:
import ollama

# Comprobación rápida para decidir si hace falta pedirle a Qwen3 que revise su propia respuesta.
# Si todo está bien, devolvemos False y nos ahorramos una segunda llamada al modelo.
def necesita_reverificacion(respuesta, metas, pregunta):
    texto = limpiar_texto(respuesta)
    # 1) Si salió demasiado corta o dejó sin cerrar el bloque interno <think>
    if len(texto.strip()) < 40 or "<think>" in texto:
        return True, "respuesta incompleta o con bloque de pensamiento"
    # 2) Si alguna de las recetas recuperadas tiene alérgenos, la respuesta debe avisar de ellos
    hay_alergenos = any(m["tiene_alergenos"] for m in metas)
    palabras_aviso = ["alergeno", "gluten", "lactosa", "huevo", "frutos", "pescado", "marisco", "soja", "sesamo", "mostaza", "apio"]
    if hay_alergenos and not any(p in texto for p in palabras_aviso):
        return True, "falta indicar los alérgenos"
    # 3) Control extra por si pidió sin gluten pero ninguna receta recuperada lo cumplía
    if "sin gluten" in limpiar_texto(pregunta) and all(m["gluten"] for m in metas):
        return True, "ninguna receta cumple sin gluten"
    return False, ""

def asistente(pregunta, filtros=None, verbose=False):
    # Si no nos pasan filtros a mano, los deducimos automáticamente de la pregunta
    if filtros is None:
        filtros = filtros_de(pregunta)

    res = buscar(pregunta, k=3, filtros=filtros)
    docs = res["documents"][0]
    metas = res["metadatas"][0]

    if not docs:
        return "No he encontrado ninguna receta en la base de datos que cumpla esos filtros."

    # Juntamos las 3 recetas separadas por --- para que Qwen3 las lea como contexto único
    contexto = "\n\n---\n\n".join(docs)
    # /no_think desactiva el razonamiento largo de Qwen3 para que responda directo
    prompt = f"""Eres un asistente de cocina claro y directo. Responde en español usando SOLO las recetas del contexto.
Elige la receta que mejor encaje con la pregunta, resume sus ingredientes y pasos, e indica siempre sus alérgenos.
Si ninguna receta del contexto encaja bien, dilo claramente sin inventar otra.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta} /no_think"""

    resp = ollama.chat(model="qwen3:1.7b", messages=[{"role": "user", "content": prompt}])
    # Por seguridad limpiamos cualquier bloque <think>...</think> (re.S permite que el punto capture saltos de línea)
    respuesta = re.sub(r"<think>.*?</think>", "", resp["message"]["content"], flags=re.S).strip()

    # Solo hacemos una segunda llamada al modelo si falla alguna comprobación
    reverificar, motivo = necesita_reverificacion(respuesta, metas, pregunta)
    if reverificar:
        if verbose:
            print("[Reverificando:", motivo, "]")
        prompt_fix = f"""Revisa y corrige la siguiente respuesta usando ÚNICAMENTE las recetas del contexto.
Motivo de revisión: {motivo}.
Asegúrate de responder en español, no inventar ingredientes y mencionar explícitamente los alérgenos detectados.

CONTEXTO:
{contexto}

RESPUESTA ANTERIOR:
{respuesta} /no_think"""
        resp2 = ollama.chat(model="qwen3:1.7b", messages=[{"role": "user", "content": prompt_fix}])
        respuesta = re.sub(r"<think>.*?</think>", "", resp2["message"]["content"], flags=re.S).strip()
    elif verbose:
        print("[OK a la primera, sin reverificación]")

    return respuesta

print(asistente("Quiero una cena vegana fácil con garbanzos", verbose=True))
print("\n" + "=" * 50 + "\n")
print(asistente("Postre sin gluten y sin huevo", verbose=True))

[OK a la primera, sin reverificación]
**Receta: ensalada de garbanzos con espinacas**  
**Alérgenos detectados: ninguno**  

**Ingredientes:**  
- 100 gramos de garbanzos cocidos  
- 1 cebolla chica  
- 100 gramos de espinaca  
- 1 cucharada sopera de aceite  
- 1 cucharada postre de aceto balsámico  

**Pasos:**  
1. **Preparación de la cebolla:** Pela y corte en pluma. Hervíe en agua hirviendo por 5 minutos, enjuágala y repite el proceso 2 veces para suavizar el sabor.  
2. **Preparación de los garbanzos:** Enjuaga los garbanzos (si son secos, remoja 12 horas en heladera y hervirás en agua limpia). Pelar o no pelar según prefieras.  
3. **Mezcla de ingredientes:** Superpón las hojas de espinaca lavadas y cortadas en tiras. Agrega los garbanzos, la cebolla y mezcla con el aceite y el aceto balsámico. Puedes reemplazar el aceto por vinagre o jugo de limón.  

**Nota:** Esta ensalada es saludable, baja en dificultad y perfecta para una cena vegana. 🥗🌱


[OK a la primera, sin reverificac

## 5. Interfaz web con Flask

Servidor web local en `http://localhost:5000` conectado a `asistente()` (con detección automática de filtros). Al ejecutar la celda quedará escuchando hasta que detengas el kernel.

In [ ]:
from flask import Flask, request

app = Flask("recetia")

HTML = """
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <title>RecetIA — Asistente de cocina</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 720px; margin: 40px auto; padding: 0 16px; color: #222; }
    .buscador { display: flex; gap: 8px; margin: 16px 0; }
    input { flex: 1; padding: 10px 12px; font-size: 16px; border: 1px solid #ccc; border-radius: 6px; }
    button { padding: 10px 18px; font-size: 16px; background: #2d6cdf; color: #fff; border: none; border-radius: 6px; cursor: pointer; }
    button:hover { background: #1f55b8; }
    pre { background: #f6f8fa; padding: 16px; border-radius: 8px; white-space: pre-wrap; line-height: 1.5; }
  </style>
</head>
<body>
  <h1>RecetIA — Asistente de cocina</h1>
  <p>Pregunta por ingredientes, tipo de plato o restricciones (vegano, sin gluten, sin lactosa, sin huevo...).</p>
  <div class="buscador">
    <input id="q" placeholder="Ej: cena vegana fácil con garbanzos" onkeydown="if(event.key==='Enter') preguntar()">
    <button onclick="preguntar()">Preguntar</button>
  </div>
  <pre id="r">Escribe una pregunta para empezar.</pre>
  <script>
    async function preguntar() {
      const texto = q.value.trim();
      if (!texto) return;
      r.textContent = "Buscando recetas y generando respuesta...";
      // encodeURIComponent codifica espacios y tildes para que viajen seguros en la URL
      const res = await fetch("/preguntar?q=" + encodeURIComponent(texto));
      r.textContent = await res.text();
    }
  </script>
</body>
</html>
"""

# Ruta principal: devuelve la página HTML al abrir http://localhost:5000
@app.route("/")
def inicio():
    return HTML

# Ruta de consulta: recibe ?q=... desde el navegador y devuelve la respuesta del asistente
@app.route("/preguntar")
def preguntar():
    pregunta = request.args.get("q", "").strip()
    if not pregunta:
        return "Escribe una pregunta primero."
    return asistente(pregunta)

# use_reloader=False evita que Flask intente reiniciar el proceso dos veces dentro de Jupyter
app.run(port=5000, use_reloader=False)

## 6. Evaluación final de las respuestas (BLEU y fidelidad)

Al final del todo medimos si las respuestas que genera **Qwen3** son buenas de verdad usando tres métricas muy sencillas:

1. **BLEU (1-2 gramas):** mide cuántas palabras y pares de palabras de la receta original aparecen en la respuesta generada. *(Como Qwen3 resume con sus propias palabras y la receta original es más larga, un BLEU moderado es completamente normal.)*
2. **Cobertura de ingredientes:** porcentaje de palabras clave de los ingredientes reales que aparecen en la respuesta.
3. **Aviso de alérgenos:** comprueba si la respuesta menciona los alérgenos detectados en la receta elegida.

In [7]:
import math
from collections import Counter

def bleu_simple(referencia, respuesta):
    """BLEU simplificado (1 y 2 gramas) sin librerías externas."""
    ref = limpiar_texto(referencia).split()
    hip = limpiar_texto(respuesta).split()
    if not ref or not hip:
        return 0.0

    # Precisión de unigramas (palabras sueltas) y bigramas (pares seguidos)
    p1 = sum((Counter(hip) & Counter(ref)).values()) / len(hip)
    big_hip = list(zip(hip, hip[1:]))
    big_ref = list(zip(ref, ref[1:]))
    p2 = sum((Counter(big_hip) & Counter(big_ref)).values()) / max(1, len(big_hip))

    # Media geométrica + penalización si la respuesta es mucho más corta que la receta
    bp = min(1.0, math.exp(1 - len(ref) / len(hip)))
    return bp * math.sqrt(max(p1, 1e-9) * max(p2, 1e-9))

def cobertura_ingredientes(doc_receta, respuesta):
    """Porcentaje de palabras clave de los ingredientes presentes en la respuesta."""
    linea_ing = ""
    for linea in doc_receta.splitlines():
        if linea.startswith("Ingredientes:"):
            linea_ing = linea.replace("Ingredientes:", "")
            break
    palabras_ing = {p for p in limpiar_texto(linea_ing).split() if len(p) >= 4}
    if not palabras_ing:
        return 1.0
    palabras_resp = set(limpiar_texto(respuesta).split())
    return len(palabras_ing & palabras_resp) / len(palabras_ing)

def aviso_alergenos_ok(meta, respuesta):
    """Comprueba si Qwen3 menciona los alérgenos de la receta (o indica que no tiene)."""
    alergenos = meta.get("alergenos", "ninguno")
    resp_limpia = limpiar_texto(respuesta)
    if alergenos == "ninguno":
        return "ningun" in resp_limpia or "sin alergen" in resp_limpia or "alergen" in resp_limpia
    lista = [a.strip().replace("_", " ") for a in alergenos.split(",")]
    return any(a in resp_limpia for a in lista)

preguntas_eval = [
    "Cena vegana con lentejas",
    "Pescado al horno sin lactosa",
    "Bizcocho o tarta sin huevo",
    "Ensalada con aguacate y tomate",
]

resultados_eval = []
for pregunta in preguntas_eval:
    res = buscar(pregunta, k=3)
    respuesta = asistente(pregunta)

    # Tomamos la mejor receta (o la más cercana) como referencia principal
    doc_top1 = res["documents"][0][0]
    meta_top1 = res["metadatas"][0][0]

    b = bleu_simple(doc_top1, respuesta)
    cob = cobertura_ingredientes(doc_top1, respuesta)
    alerg_ok = aviso_alergenos_ok(meta_top1, respuesta)
    resultados_eval.append((pregunta, b, cob, alerg_ok))

    print(f"Pregunta: {pregunta}")
    print(f"  -> BLEU: {b:.3f} | Cobertura ingredientes: {cob:.1%} | Aviso alérgenos: {'OK' if alerg_ok else 'Revisar'}")

bleu_medio = sum(r[1] for r in resultados_eval) / len(resultados_eval)
cob_media = sum(r[2] for r in resultados_eval) / len(resultados_eval)
alerg_medio = sum(1 for r in resultados_eval if r[3]) / len(resultados_eval)

print("\n--- RESUMEN FINAL DE GENERACIÓN ---")
print(f"BLEU medio (1-2 gramas):   {bleu_medio:.3f}")
print(f"Cobertura ingredientes:    {cob_media:.1%}")
print(f"Aviso correcto alérgenos:  {alerg_medio:.1%}")

Pregunta: Cena vegana con lentejas
  -> BLEU: 0.042 | Cobertura ingredientes: 100.0% | Aviso alérgenos: Revisar
Pregunta: Pescado al horno sin lactosa
  -> BLEU: 0.020 | Cobertura ingredientes: 100.0% | Aviso alérgenos: OK
Pregunta: Bizcocho o tarta sin huevo
  -> BLEU: 0.004 | Cobertura ingredientes: 100.0% | Aviso alérgenos: OK
Pregunta: Ensalada con aguacate y tomate
  -> BLEU: 0.048 | Cobertura ingredientes: 100.0% | Aviso alérgenos: OK

--- RESUMEN FINAL DE GENERACIÓN ---
BLEU medio (1-2 gramas):   0.028
Cobertura ingredientes:    100.0%
Aviso correcto alérgenos:  75.0%


### Interpretación rápida de los resultados

- **BLEU medio (1-2 gramas): `0.028`** — Es bajo porque BLEU compara palabras exactas contra toda la receta original (mucho más larga), mientras que Qwen3 **resume** los pasos con sus propias palabras.
- **Cobertura de ingredientes: `100.0%`** — El modelo incluye todos los ingredientes clave de la receta recuperada sin omitirlos ni inventarlos.
- **Aviso de alérgenos: `75.0%` (3 de 4)** — Cuando la receta tiene alérgenos los avisa correctamente; en la receta sin alérgenos a veces deja el campo vacío en lugar de escribir "ninguno".